In [1]:

# =============================================================================
# CELL 1 – IMPORTS, DARK THEME, AND HELPERS
# =============================================================================

import importlib
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML, Markdown

# -------------------- Dark Theme --------------------
display(HTML("""
<style>
    body, .jp-Notebook, .jp-OutputArea-output, .jp-RenderedHTMLCommon {
        background-color: #1e1e1e !important;
        color: #d4d4d4 !important;
    }
    h2, h3, h4 {
        color: #4fc3f7 !important;
        border-bottom: 2px solid #3498db !important;
        padding-bottom: 4px;
    }
    b, strong { color: #f48fb1 !important; }
    .highlight {
        background-color: #2d2d2d !important;
        padding: 10px;
        border-left: 4px solid #3498db;
        margin: 4px 0;
        color: #d4d4d4;
    }
    code {
        background-color: #333 !important;
        color: #ffcc80 !important;
        padding: 2px 4px;
        border-radius: 4px;
    }
    .dataframe {
        background-color: #2d2d2d !important;
        color: #d4d4d4 !important;
    }
</style>
"""))

# -------------------- Verbosity Flags --------------------
SHOW_VERBOSE = True
SHOW_INFO = True
SHOW_CRITICAL = True
SHOW_DEBUG = True
# For demonstration, we keep them False to reduce output; you can set True as needed.


# -------------------- Import Extraction Module --------------------
import Extraction6_new as Extraction6
importlib.reload(Extraction6)

# -------------------- Helper Functions --------------------
def display_title(title: str):
    display(HTML(f"<h2>{title}</h2>"))

def display_info(message: str):
    display(HTML(f"<div class='highlight'>{message}</div>"))
    

def discover_extractions(exp_root: Path) -> pd.DataFrame:
    rows = []
    for f in exp_root.rglob("extraction.json"):
        parts = f.relative_to(exp_root).parts
        try:
            models_idx = parts.index("models")
            datasets_idx = parts.index("datasets")
            model = "/".join(parts[models_idx+1:datasets_idx])
            dataset = parts[datasets_idx+1]
            rows.append({"model": model, "dataset": dataset, "path": str(f)})
        except ValueError:
            rows.append({"model": None, "dataset": None, "path": str(f)})
    return pd.DataFrame(rows)

def load_experiment_results(exp_root: Path) -> pd.DataFrame:
    """
    Load all extraction results from the experiment directory, inferring model and
    dataset names from the file path if they are missing in the JSON.
    """
    records = []
    exp_root = Path(exp_root)
    if not exp_root.exists():
        return pd.DataFrame()

    for meta_file in exp_root.rglob("extraction.json"):
        try:
            with open(meta_file, "r") as f:
                meta = json.load(f)

            # Extract model and dataset from path (most reliable)
            parts = meta_file.relative_to(exp_root).parts
            try:
                models_idx = parts.index("models")
                datasets_idx = parts.index("datasets")
                model = "/".join(parts[models_idx+1:datasets_idx])
                dataset = parts[datasets_idx+1]
            except ValueError:
                # Fallback to metadata
                model = meta.get("model", {}).get("name")
                dataset = meta.get("dataset", {}).get("name")

            record = {
                "experiment_id": meta.get("experiment_id"),
                "model": model,
                "dataset": dataset,
                "status": meta.get("status"),
                "completed_samples": meta.get("performance", {}).get("completed_samples"),
                "total_samples": meta.get("dataset", {}).get("samples"),
                "batch_size": meta.get("extraction", {}).get("batch_size"),
                "pooling": meta.get("extraction", {}).get("pooling"),
                "max_length": meta.get("extraction", {}).get("max_length"),
                "samples_per_second": meta.get("performance", {}).get("samples_per_second"),
                "tokens_per_second": meta.get("performance", {}).get("tokens_per_second"),
                "elapsed_seconds": meta.get("performance", {}).get("elapsed_seconds"),
                "error": None,
                "text_column": meta.get("dataset", {}).get("text_column"),
                "label_column": meta.get("dataset", {}).get("labels", {}).get("label_column"),
            }
            records.append(record)
        except Exception:
            continue

    return pd.DataFrame(records)

print("Environment ready. Dark theme applied.")

Environment ready. Dark theme applied.


In [2]:
# =============================================================================
# CELL 2 – LOAD AND INSPECT DATASETS
# =============================================================================

from Get_Go_Emo import get_go
from Get_Isear import get_isr

goEmo = get_go()
isear = get_isr()

DATASETS = {
    "goEmo": goEmo,
    "ISEAR": isear,
}

for name, df in DATASETS.items():
    display_title(f"Dataset: {name}")
    display_info(f"Shape: {df.shape}")
    display(df.head(2))
    text_col = Extraction6.detect_text_column(df, show_verbose=False)
    label_cols = [c for c in df.columns if c.lower() in ("labels", "label", "emotion", "target")]
    label_col = label_cols[0] if label_cols else None
    display_info(f"Text column: <b>{text_col}</b><br>Label column: <b>{label_col or 'None'}</b><br>Samples: <b>{len(df):,}</b>")

display_title("Unified ID Scheme")
display_info("""
Each sample is assigned a unique integer ID (0..N-1) matching its row index.
This ID links:
• Input text (original DataFrame index)
• Hidden state vector (row in hidden_states.npy)
• Label (row in labels.npy)
No shuffling occurs, guaranteeing one‑to‑one mapping.
""")

,labels,clean_text
0,[27],my favourite food is anything i didnt have to ...
1,[27],"now if he does off himself, everyone will thin..."


,clean_text,labels
0,during the period of falling in love each time...,1
1,when i was involved in a traffic accident,2


In [3]:
# =============================================================================
# CELL 3 – RUN MODEL MATRIX (ENHANCED, WITH SAFEGUARDS)
# =============================================================================

# This call will automatically resume from existing completions.
# If interrupted, simply re-run this cell to continue.
results = Extraction6.run_model_matrix(
    datasets=DATASETS,
    groups=None,                     # all 25 models
    base_output="/Volumes/Amirali/hidden_states",
    pooling="mean",
    max_length=512,
    use_half_precision=True,
    auto_batch_size=False,
    flush_every_batches=8,
    continue_on_model_error=True,
    show_verbose=SHOW_VERBOSE,
    show_info=SHOW_INFO,
    show_critical=SHOW_CRITICAL,
    show_debug=SHOW_DEBUG,
)

print(f"Returned result records: {len(results)}")


╔══════════════════════════════════════════════════════════════════════════════════════╗
║ MODEL MATRIX                                                                         ║
╚══════════════════════════════════════════════════════════════════════════════════════╝
  Requested experiment : baseline_v5_001
  Transformers         : 4.47.1
  PyTorch              : 2.2.2
  Models               : 25
  Datasets             : 2
  Runtime batch tuning : DISABLED
  Output root          : /Volumes/Amirali/hidden_states
01. [01_encoders] BERT         0.11B  google-bert/bert-base-uncased
02. [01_encoders] DistilBERT   0.066B  distilbert/distilbert-base-uncased
03. [01_encoders] RoBERTa      0.125B  FacebookAI/roberta-base
04. [01_encoders] ELECTRA      0.014B  google/electra-small-discriminator
05. [01_encoders] DeBERTa      0.14B  microsoft/deberta-v3-small
06. [02_early_decoders] GPT          0.124B  gpt2
07. [02_early_decoders] GPT-Neo      0.125B  EleutherAI/gpt-neo-125m
08. [02_early_decode

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

  ✓ Snapshot ready
    Attempt time        : 1.36s
    Total preparation   : 1.40s
    Checkpoint size     : 0.251 GiB
    Snapshot            : /Volumes/Amirali/hidden_states/huggingface_cache/hub/models--HuggingFaceTB--SmolLM2-135M/snapshots/93efa2f097d58c2a74874c7e644dbc9b0cee75a2

────────────────────────────────────────────────────────────────────────────────────────
MODEL PREFLIGHT
────────────────────────────────────────────────────────────────────────────────────────
  model                             : HuggingFaceTB/SmolLM2-135M
  model_type                        : llama
  architecture                      : decoder
  dtype                             : torch.float32
  device                            : cpu
  layers                            : 30
  hidden_size                       : 576
  hidden_states                     : 31
  max_length                        : 512
  checkpoint                        : {'active_bytes': 269064648, 'active_gb': 0.25058598071336746, 'inve

ISEAR:   0%|          | 0/7666 [00:00<?, ?sample/s]

[start] starting at absolute sample 0
[batch] 0:64 processed
[batch] 64:128 processed

────────────────────────────────────────────────────────────────────────────────────────
RUNTIME / SPEED TELEMETRY :: ISEAR
────────────────────────────────────────────────────────────────────────────────────────
  Progress             : 128/7,666 (1.67%)
  Newly computed       : 128
  Wall time            : 21.89s
  Throughput           : 5.85 samples/s
  ETA                  : 21m 29.0s
  Current stage        : measurement

  LAST BATCH
    Range              : 64:128
    New samples        : 64
    Total              : 13.40s
    Forward            : 12.67s
    Tokenization       : 0.02s
    Input transfer     : 0.00s
    Pooling            : 0.13s
    Conversion         : 0.00s
    Memmap write       : 0.09s
    Flush              : 0.10s
    Sequence length    : 59
    Tokens             : 1283
    Token throughput   : 95.73 tokens/s

  ROLLING PERFORMANCE
    Mean batch         : 10.910s
    Me

...

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

  ✓ Snapshot ready
    Attempt time        : 5m 49.0s
    Total preparation   : 5m 49.0s
    Checkpoint size     : 2.875 GiB
    Snapshot            : /Volumes/Amirali/hidden_states/huggingface_cache/hub/models--Qwen--Qwen2.5-1.5B/snapshots/8faed761d45a263340a0528343f099c05c9a4323

────────────────────────────────────────────────────────────────────────────────────────
MODEL PREFLIGHT
────────────────────────────────────────────────────────────────────────────────────────
  model                             : Qwen/Qwen2.5-1.5B
  model_type                        : qwen2
  architecture                      : decoder
  dtype                             : torch.float32
  device                            : cpu
  layers                            : 28
  hidden_size                       : 1536
  hidden_states                     : 29
  max_length                        : 512
  checkpoint                        : {'active_bytes': 3087471240, 'active_gb': 2.875431664288044, 'inventory': {'sa

goEmo:   0%|          | 0/54263 [00:00<?, ?sample/s]

[start] starting at absolute sample 0
[batch] 0:8 processed

────────────────────────────────────────────────────────────────────────────────────────
RUNTIME / SPEED TELEMETRY :: goEmo
────────────────────────────────────────────────────────────────────────────────────────
  Progress             : 8/54,263 (0.01%)
  Newly computed       : 8
  Wall time            : 38.73s
  Throughput           : 0.21 samples/s
  ETA                  : 72h 56m 50.4s
  Current stage        : measurement

  LAST BATCH
    Range              : 0:8
    New samples        : 8
    Total              : 38.32s
    Forward            : 37.38s
    Tokenization       : 0.11s
    Input transfer     : 0.00s
    Pooling            : 0.09s
    Conversion         : 0.01s
    Memmap write       : 0.14s
    Flush              : 0.08s
    Sequence length    : 37
    Tokens             : 136
    Token throughput   : 3.55 tokens/s

  ROLLING PERFORMANCE
    Mean batch         : 38.316s
    Median batch       : 38.316s
    

In [ ]:
# =============================================================================
# CELL 4 – COLLECT ALL RESULTS FROM DISK (PARTIAL INCLUDED)
# =============================================================================

exp_root = Path("/Volumes/Amirali/hidden_states/experiments/baseline_v5_001")
df_results = load_experiment_results(exp_root)

if not df_results.empty:
    display_title("Extraction Progress")
    display(df_results[['model', 'dataset', 'status', 'completed_samples', 'total_samples']].head(15))
    display_info(f"Total dataset‑model pairs found: <b>{len(df_results)}</b>")
else:
    display_info("No extraction metadata found yet. Run the extraction first (Cell 3).")

In [ ]:
# =============================================================================
# CELL 4.5 – FULL EXPERIMENT AUDIT & VALIDATION
# =============================================================================

from IPython.display import display, HTML
import pandas as pd
import numpy as np

# Path to your experiment
exp_root = Path("/Volumes/Amirali/hidden_states/experiments/baseline_v5_001")

display_title("🔍 Automated Audit & Validation")
display_info("Scanning all model‑dataset pairs and verifying data integrity...")

# Run the audit – this will automatically discover all pairs, validate checksums and sample IDs,
# and optionally fix issues. Set fix_issues=False if you only want a report.
audit_results = Extraction6.audit_experiment(
    experiment_root=exp_root,
    datasets=DATASETS,          # provides the original datasets for sample ID validation
    show_details=False,         # set True if you want per‑dataset detailed prints
    fix_issues=True,            # automatically correct checksum and sample ID mismatches
)

# Convert audit results to a DataFrame for easy viewing
df_audit = pd.DataFrame(audit_results)

# Keep only relevant columns for display
display_cols = [
    "model_name", "dataset_name", "n_samples", "completed_count", "status",
    "checksum_match", "sample_ids_match"
]
if not df_audit.empty:
    df_display = df_audit[display_cols].copy()

    # Clean up status strings and add icons
    status_icons = {
        "complete": "✅",
        "partial": "⏳",
        "missing_files": "❌",
        "missing_metadata": "⚠️",
    }
    df_display["status_icon"] = df_display["status"].map(status_icons).fillna("❓")

    # Boolean columns to checkmarks
    def bool_to_icon(val):
        if val is True:
            return "✅"
        elif val is False:
            return "❌"
        else:
            return "—"

    df_display["checksum"] = df_display["checksum_match"].apply(bool_to_icon)
    df_display["sample_ids"] = df_display["sample_ids_match"].apply(bool_to_icon)

    # Reorder columns for a clean table
    final_cols = ["model_name", "dataset_name", "n_samples", "completed_count", "status_icon", "checksum", "sample_ids"]
    df_final = df_display[final_cols].rename(columns={
        "model_name": "Model",
        "dataset_name": "Dataset",
        "n_samples": "Total Samples",
        "completed_count": "Completed",
        "status_icon": "Status",
        "checksum": "Checksum",
        "sample_ids": "Sample IDs",
    })

    # Create a styled HTML table with custom colors
    def color_status(val):
        if val == "✅":
            return "color: #4caf50;"
        elif val == "⏳":
            return "color: #ff9800;"
        elif val == "❌":
            return "color: #f44336;"
        elif val == "⚠️":
            return "color: #ffeb3b;"
        return ""

    # Display the table
    display(HTML("<h3>Audit Summary – All Model‑Dataset Pairs</h3>"))
    styled_df = df_final.style.applymap(color_status, subset=["Status"])
    display(styled_df)

    # Summary statistics
    total_pairs = len(df_audit)
    complete = (df_audit["status"] == "complete").sum()
    partial = (df_audit["status"] == "partial").sum()
    missing = (df_audit["status"].isin(["missing_files", "missing_metadata"])).sum()
    checksum_ok = df_audit["checksum_match"].sum() if "checksum_match" in df_audit else 0
    ids_ok = df_audit["sample_ids_match"].sum() if "sample_ids_match" in df_audit else 0
    actions_taken = sum(len(r.get("actions_taken", [])) for r in audit_results)

    display_info(f"""
    <b>Total pairs scanned:</b> {total_pairs} &nbsp;|&nbsp;
    <b>✅ Complete:</b> {complete} &nbsp;|&nbsp;
    <b>⏳ Partial:</b> {partial} &nbsp;|&nbsp;
    <b>❌ Missing/Error:</b> {missing} &nbsp;|&nbsp;
    <b>🔧 Checksum OK:</b> {checksum_ok} &nbsp;|&nbsp;
    <b>🔧 Sample IDs OK:</b> {ids_ok} &nbsp;|&nbsp;
    <b>🛠️ Issues fixed:</b> {actions_taken}
    """)

    # Optionally print model-level summary (which models are fully complete)
    model_summary = df_audit.groupby("model_name").apply(
        lambda g: pd.Series({
            "total_datasets": len(g),
            "complete": (g["status"] == "complete").sum(),
            "partial": (g["status"] == "partial").sum(),
            "missing": (g["status"].isin(["missing_files", "missing_metadata"])).sum(),
        })
    ).reset_index()
    model_summary.columns = ["Model", "Datasets", "Complete", "Partial", "Missing/Error"]

    display(HTML("<h3>Per‑Model Completeness</h3>"))
    display(model_summary)

else:
    display_info("No extraction data found to audit.")

In [ ]:
# =============================================================================
# CELL 5 – VISUALISATIONS & ANALYSIS
# =============================================================================

if not df_results.empty:
    # Prepare data for plotting
    df = df_results.copy()
    df['completion_pct'] = df['completed_samples'] / df['total_samples'] * 100
    df['status_clean'] = df['status'].replace({'complete': 'Complete', 'partial': 'Partial', 'failed': 'Failed', 'already_complete': 'Already Complete'})

    sns.set_style("darkgrid")
    plt.rcParams.update({
        'figure.facecolor': '#1e1e1e',
        'axes.facecolor': '#2d2d2d',
        'axes.edgecolor': '#d4d4d4',
        'axes.labelcolor': '#d4d4d4',
        'text.color': '#d4d4d4',
        'xtick.color': '#d4d4d4',
        'ytick.color': '#d4d4d4',
        'grid.color': '#444444',
        'legend.facecolor': '#2d2d2d',
        'legend.edgecolor': '#d4d4d4',
    })

    # ---- 1. Completion status per model/dataset ----
    fig, ax = plt.subplots(figsize=(12, 8))
    pivot = df.pivot_table(index='model', columns='dataset', values='completion_pct', aggfunc='max')
    sns.heatmap(pivot, annot=True, fmt=".0f", cmap="viridis", cbar_kws={'label': 'Completion %'}, ax=ax)
    ax.set_title('Completion Percentage by Model and Dataset', color='#4fc3f7')
    plt.tight_layout()
    plt.show()

    # ---- 2. Throughput (samples/sec) by model ----
    df_complete = df[df['samples_per_second'].notna()]
    if not df_complete.empty:
        fig, ax = plt.subplots(figsize=(14, 6))
        sns.barplot(data=df_complete, x='model', y='samples_per_second', hue='dataset', palette='coolwarm', ax=ax)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
        ax.set_title('Extraction Throughput (samples/sec)', color='#4fc3f7')
        plt.tight_layout()
        plt.show()

    # ---- 3. Total time per model ----
    df_time = df.groupby('model')['elapsed_seconds'].sum().reset_index().sort_values('elapsed_seconds', ascending=False)
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.barplot(data=df_time, x='elapsed_seconds', y='model', palette='magma', ax=ax)
    ax.set_xlabel('Total Elapsed Time (seconds)')
    ax.set_title('Cumulative Extraction Time per Model', color='#4fc3f7')
    plt.tight_layout()
    plt.show()

    # ---- 4. Label coverage for completed datasets ----
    # We'll just display label column info
    display_title("Label Columns Used")
    display(df[['model', 'dataset', 'label_column']].drop_duplicates())
else:
    display_info("No data to visualise. Please run extraction first.")

In [ ]:
# =============================================================================
# CELL 6 – FINAL SUMMARY & EXPORT
# =============================================================================

if not df_results.empty:
    # Save consolidated CSV
    output_csv = exp_root / "extraction_summary.csv"
    df_results.to_csv(output_csv, index=False)
    display_title("Final Report")
    display(df_results)
    display_info(f"Report exported to <code>{output_csv}</code>")
else:
    display_info("Nothing to export yet.")

In [ ]:
discover_extractions(exp_root)

In [ ]:
# Check which models have at least one dataset complete
model_status = df_results.groupby("model")["dataset"].nunique()
print(model_status)

# 🚀 PROJECT REVIEW – EMOTION LEARNING IN LLMs
## `·` State‑of‑‑the‑Art Feedback & Action Plan `·`


# 1. EXECUTIVE SUMMARY

| Aspect | Status | Trend |
|--------|--------|-------|
| Code quality | ⭐⭐⭐⭐⭐ | Exceptional |
| Experimental rigour | ⭐⭐⭐⭐ | Solid controls in place |
| Result interpretation | ⭐⭐ | Needs narrative depth |
| Specification alignment | ⚠️ | QWEN de‑prioritised |
| Literature integration | ⭐⭐⭐ | More explicit links needed |

> ✅ **Overall:** A **diamond in the rough**. Your extraction/probe infrastructure is world‑class, but you’ve over‑invested in the tool and under‑invested in the question. **Refocus on QWEN** and **close the analytical loop** to secure a strong First.


# 2. CRITICAL GAPS & FIXES

## 🔴 1. QWEN is missing from your results
**Spec says:** *“The main focus is QWEN.”*  
**Current reality:** You run 25 small models, QWEN is not the primary focus.

**Fix:**  
- Prioritise extracting **QWEN2‑0.5B** and **QWEN2‑1.5B** (they work on CPU).  
- Acknowledge the limitation honestly in the dissertation.  
- Frame the 25‑model matrix as a **supplementary contribution**, not the core result.

## 🟠 2. Analysis is descriptive, not explanatory
You extract layers and compute F1, but you don’t interpret *why* certain layers excel.

**Fix:**  
- Write a narrative: *“Layer 3 excels because it encodes syntactic patterns correlated with emotional valence.”*  
- Compare to known literature (e.g., Tenney et al. on BERT).  
- Add **ablation studies** (remove emotion words, re‑run probes → see the drop).

## 🟡 3. Performance bottleneck
Extraction runs on CPU, very slow (4–5 hours for small models). QWEN‑7B is impossible.

**Fix:**  
- Acknowledge this as a **constraint** in your write‑up.  
- Frame your codebase as **optimised for minimal resources** – a contribution in itself.  
- If feasible, use **cloud GPU** for a few days to get the critical QWEN data.

## 🟢 4. Logbook: event‑centric, not reflection‑centric
You record *what* you did, but not *what you learned*.

**Fix:**  
- Add “Key learnings” after each work block.  
- Reflect on failed experiments.  
- Document your pivot decisions (e.g., why you switched from JSON to `.npy`).


# 3. ACTION PLAN

## 📌 Phase 1 – Immediate (next 2 weeks)
- [ ] **Run extraction** for:
  - `Qwen/Qwen2-0.5B`
  - `Qwen/Qwen2-1.5B`
  - `Qwen/Qwen2.5-0.5B` (optional)
- [ ] **Run the full probe analysis** on these QWEN states.
- [ ] **Start ablation study**: remove emotion keywords from ISEAR and re‑probe.

## 📌 Phase 2 – Mid‑term (4–6 weeks)
- [ ] **Deepen literature integration** – explicitly link your findings to prior work on BERT and other LLMs.
- [ ] **Add interpretability technique** – e.g., attention visualisation (BertViz) or LIME.
- [ ] **Write the “So what?” narrative** – what are the implications for transfer learning and emotion recognition?

## 📌 Phase 3 – Pre‑submission
- [ ] **Final write‑up structure**: Introduction → Lit Review → Methodology → Results → Discussion → Conclusion.
- [ ] **Unified dashboard**: heatmaps, layer curves, model‑comparison charts, ablation results.
- [ ] **Polish code**: `README.md`, `requirements.txt`, user guide for reproducibility.


# 4. WEEKLY SCHEDULE SUGGESTION (7‑day Power Week)

| Day | Morning (09:00–12:00) | Afternoon (13:00–16:00) | Evening (17:00–19:00) |
|-----|------------------------|--------------------------|------------------------|
| Mon | **Exam Module 1** – Active recall | **Exam Module 2** – Practice questions | **Attendance Module A** – Prep |
| Tue | **Exam Module 3** – Concept mapping | **QWEN extraction** (background) | **Attendance Module B** – Follow‑up |
| Wed | **Exam Module 1** – Past papers | **Data analysis** – Probe results | **Project** – Write‑up |
| Thu | **Exam Module 2** – Flashcards | **Project** – Ablation design | **Rest / Light reading** |
| Fri | **Exam Module 3** – Problem sets | **Project** – Visualisation | **Review week’s progress** |
| Sat | **Consolidation** – All modules (weak areas) | **Project** – Code cleanup | **Social / Recovery** |
| Sun | **Rest / Planning** – No academic work | **Rest / Planning** – No academic work | **Rest / Planning** – No academic work |

> **Start‑day protocol:** 30 min walk → 15 min planning → 1st focus block.  
> **2‑week review:** Ask: *“Am I on track? Is this technique effective? Is my health okay?”*


# 5. FIRST‑CLASS BLUEPRINT (Grade Targets)

| Module Type | Target Grade | Strategy |
|-------------|--------------|----------|
| Exam Module 1 | 72–75% | Past papers, spaced repetition |
| Exam Module 2 | 70–73% | Flashcards, peer teaching |
| Exam Module 3 | 68–72% | Concept maps, problem sets |
| Attendance A | 75–80% | Lead seminars, engage tutor |
| Attendance B | 72–78% | Exceptional coursework, early drafts |

> Current average: 68% → need ~73% across all to secure a strong First.


# 6. KEY RESOURCES & TOOLS

- **Dataset:** GoEmotions + ISEAR (already preprocessed)
- **Models:** QWEN2‑0.5B, 1.5B (primary); BERT (control); 25‑model matrix (supplement)
- **Probe code:** `unified_hidden_state_probe_v4_2.py` (state‑of‑the‑art)
- **Extraction:** `Extraction6_new.py` (resume‑capable, manifest‑tracking)
- **Visualisation:** `Analyser_.py` + `Analyzer_Visualizer_.ipynb`
- **Explainability:** Consider adding **BertViz** or **LIME** for attention maps


# 7. FINAL CHECKLIST

- [ ] QWEN extraction completed
- [ ] QWEN probe analysis done
- [ ] Ablation study executed
- [ ] Attention visualisation added
- [ ] Literature integrated
- [ ] Final write‑up structured
- [ ] Code documented and shareable

> 🏆 **Remember:** The difference between 68% and 78% is **research maturity** – synthesis, self‑critique, and forward thinking. You have the technical skills. Now build the intellectual scaffolding around them.

**You’ve got this.** 🚀